# Solution 1
Normal way to process: GET the pdf file then convert into image (jpg) to process.

This solution works with รายการเลือกตั้งแบบ**บัญชีรายชื่อ**

## Set up & Import

In [ ]:
import os
import re
import json
import tempfile
import time
from pathlib import Path
from dotenv import load_dotenv
import fitz
from thefuzz import fuzz
from typhoon_ocr import ocr_document
from IPython.display import display, Markdown
import tempfile
from pdf2image import convert_from_path
from PIL import Image

load_dotenv()
TYPHOON_API_KEY = os.getenv("TYPHOON_OCR_API_KEY") or os.getenv("TYPHOON_API_KEY")
if not TYPHOON_API_KEY:
    print("⚠️ ไม่พบ TYPHOON_API_KEY กรุณาตรวจสอบไฟล์ .env")
else:
    print("✅ System Ready: Loaded API Key successfully.")

MASTER_CANDIDATES = [
    "นายอรรถพล โต๋วสัจจา",
    "นายชาดา ไทยเศรษฐ์",
    "นางสาวสุชาดา บัวพันธ์",
    "ร.ต.ศึกษา ฟุ้งเฟื่อง ร.น.",
    "นางสุพรรณษา นันทา"
]

MASTER_PARTIES = [
    "ไทยทรัพย์ทวี", "เพื่อชาติไทย", "ใหม่", "มิติใหม่", "รวมใจไทย", 
    "รวมไทยสร้างชาติ", "พลวัต", "ประชาธิปไตยใหม่", "เพื่อไทย", "ทางเลือกใหม่", 
    "เศรษฐกิจ", "เสรีรวมไทย", "รวมพลังประชาชน", "ท้องที่ไทย", "อนาคตไทย",
    "พลังเพื่อไทย", "ไทยชนะ", "พลังสังคมใหม่", "สังคมประชาธิปไตยไทย", "ฟิวชัน", 
    "ไทรวมพลัง", "ก้าวอิสระ", "ปวงชนไทย", "วิชชั่นใหม่", "เพื่อชีวิตใหม่", 
    "คลองไทย", "ประชาธิปัตย์", "ไทยก้าวหน้า", "ไทยภักดี", "แรงงานสร้างชาติ", 
    "ประชากรไทย", "ครูไทยเพื่อประชาชน", "ประชาชาติ", "สรางอนาคตไทย", "รักชาติ", 
    "ไทยพร้อม", "ภูมิใจไทย", "พลังธรรมใหม่", "กรีน", "ไทยธรรม", 
    "แผ่นดินธรรม", "กล้าธรรม", "พลังประชารัฐ", "โอกาสใหม่", "เป็นธรรม", 
    "ประชาชน", "ประชาไทย", "ไทยสร้างไทย", "ไทยก้าวใหม่", "ประชาอาสาชาติ",  
    "พร้อม", "เครือข่ายชาวนาแห่งประเทศไทย", "ไทยพิทักษ์ธรรม", "ความหวังใหม่", "ไทยรวมไทย", 
    "เพื่อบ้านเมือง", "พลังไทยรักชาติ" 
]

## Parser Class

In [ ]:
class ElectionOCRParser:
    def __init__(self):
        pass

    def get_page_count(self, pdf_path):
        try:
            with open(pdf_path, 'rb') as f:
                return len(PdfReader(f).pages)
        except Exception as e:
            print(f"Error reading PDF: {e}")
            return 0

    def extract_number(self, pattern, text):
        match = re.search(pattern, text)
        if match:
            return int(match.group(1).replace(',', '').strip())
        return 0

    def parse_markdown(self, markdown_text):
        data = {
            "eligible_voters": self.extract_number(r'บัญชีรายชื่อผู้มีสิทธิ.*?จำนวน\s*([\d,]+)', markdown_text),
            "voters_showed_up": self.extract_number(r'มาแสดงตน.*?จำนวน\s*([\d,]+)', markdown_text),
            "ballots_allocated": self.extract_number(r'ได้รับจัดสรร.*?จำนวน\s*([\d,]+)', markdown_text),
            "ballots_used": self.extract_number(r'บัตรเลือกตั้งที่ใช้.*?จำนวน\s*([\d,]+)', markdown_text),
            "valid_ballots": self.extract_number(r'บัตรดี.*?จำนวน\s*([\d,]+)', markdown_text),
            "invalid_ballots": self.extract_number(r'บัตรเสีย.*?จำนวน\s*([\d,]+)', markdown_text),
            "no_vote_ballots": self.extract_number(r'ไม่เลือก.*?จำนวน\s*([\d,]+)', markdown_text),
            "ballots_remaining": self.extract_number(r'บัตรเลือกตั้งที่เหลือ.*?จำนวน\s*([\d,]+)', markdown_text),
            "scores": {}
        }
        
        # 1. ลองดึงข้อมูลจากตาราง HTML (เช่น <tr><td>๑๑</td><td>พรรค...</td><td>150</td></tr>)
        html_pattern = re.compile(r'<tr>\s*<t[dh]>.*?</t[dh]>\s*<t[dh]>(.*?)</t[dh]>\s*<t[dh]>\s*([\d,]+).*?</t[dh]>\s*</tr>', re.IGNORECASE)
        matches = html_pattern.findall(markdown_text)
        
        # 2. ถ้าย้งไม่เจอ ลองดึงข้อมูลจากตาราง Markdown (รองรับเลขไทยและอารบิก)
        if not matches:
            md_pattern = re.compile(r'\|\s*[\d๑-๙]+\s*\|\s*([^|]+?)\s*\|\s*([\d,]+).*?\|')
            matches = md_pattern.findall(markdown_text)
            
        # 3. จัดเก็บลง Dictionary
        for name, score in matches:
            # ทำความสะอาดชื่อพรรค เผื่อมี Tag HTML หรือเว้นวรรคแปลกๆ ติดมา
            clean_name = re.sub(r'<[^>]+>', '', name).strip()
            
            # ข้ามแถวที่เป็นหัวตาราง (Header)
            if "ชื่อ" in clean_name and "พรรค" in clean_name:
                continue
                
            clean_score = int(score.replace(',', '').strip())
            data['scores'][clean_name] = clean_score
            
        return data

    def validate_data(self, data, file_type):
        flags = {
            "flag_math_total_used": False, 
            "flag_math_valid_score": False, 
            "flag_name_mismatch": False,    
            "flag_details": []              
        }
        
        sum_ballots = data['valid_ballots'] + data['invalid_ballots'] + data['no_vote_ballots']
        if data['ballots_used'] != sum_ballots:
            flags["flag_math_total_used"] = True
            flags["flag_details"].append(f"บัตรที่ใช้({data['ballots_used']}) != รวมบัตรย่อย({sum_ballots})")
            
        sum_scores = sum(data['scores'].values())
        if data['valid_ballots'] != sum_scores:
            flags["flag_math_valid_score"] = True
            flags["flag_details"].append(f"บัตรดี({data['valid_ballots']}) != ผลรวมตาราง({sum_scores})")
            
        reference_list = MASTER_PARTIES if file_type == "บัญชีรายชื่อ" else MASTER_CANDIDATES
        mismatched_names = []
        
        for name in data['scores'].keys():
            if not reference_list: continue
            best_match_score = max([fuzz.partial_ratio(name, master) for master in reference_list])
            if best_match_score < 85: 
                mismatched_names.append(name)
                
        if mismatched_names:
            flags["flag_name_mismatch"] = True
            flags["flag_details"].append(f"ชื่อเพี้ยน: {', '.join(mismatched_names)}")
            
        needs_manual_check = any([
            flags["flag_math_total_used"], 
            flags["flag_math_valid_score"], 
            flags["flag_name_mismatch"]
        ])
        
        flags["flag_details"] = " | ".join(flags["flag_details"]) if flags["flag_details"] else "OK"
        flags["needs_manual_check"] = needs_manual_check
        
        return flags

## Single file Runner

In [ ]:
# test_pdf_path = "../อำเภอบ้านไร่/ตำบลแก่นมะกรูด/หน่วยเลือกตั้งที่ 2/ส.ส.5ทับ18 (บช.).pdf" 
test_pdf_path = "../อำเภอบ้านไร่/ตำบลแก่นมะกรูด/หน่วยเลือกตั้งที่ 2/ส.ส.5ทับ18.pdf" 

In [ ]:
if not os.path.exists(test_pdf_path):
    print(f"❌ หาไฟล์ไม่เจอ กรุณาตรวจสอบ Path: {test_pdf_path}")
else:
    parser = ElectionOCRParser()
    file_type = "บัญชีรายชื่อ" if "บช" in os.path.basename(test_pdf_path) else "แบ่งเขต"
    
    print(f"📄 กำลังประมวลผลไฟล์: {os.path.basename(test_pdf_path)} (ประเภท: {file_type})")
    
    full_markdown = ""
    try:
        doc = fitz.open(test_pdf_path)
        total_pages = len(doc)
        
        for page_idx in range(total_pages):
            print(f"   ⏳ กำลังดึงข้อมูลจากหน้า {page_idx + 1}/{total_pages} ...")
            
            # 1. ลด DPI เหลือ 150 (เพียงพอสำหรับ LLM และประหยัดขนาดไฟล์)
            page = doc.load_page(page_idx)
            pix = page.get_pixmap(dpi=150)
            
            with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as temp_file:
                temp_path = temp_file.name
                pix.save(temp_path)
            
            # 2. บีบอัดรูปภาพด้วย PIL ให้ขนาดเบาที่สุด (Quality 75%)
            with Image.open(temp_path) as img:
                img.thumbnail((1600, 2400)) # ลิมิตความกว้าง/ยาวสูงสุด
                img.save(temp_path, "JPEG", quality=75, optimize=True)
            
            # 3. ส่งเข้า API พร้อมระบบ Retry (แก้ปัญหา 504 Timeout)
            md_text = ""
            max_retries = 3
            for attempt in range(max_retries):
                try:
                    md_text = ocr_document(pdf_or_image_path=temp_path)
                    break # ถ้าสำเร็จให้หลุดลูป retry
                except Exception as e:
                    print(f"      ⚠️ API Timeout/Error (ครั้งที่ {attempt + 1}/{max_retries}) กำลังลองใหม่...")
                    time.sleep(3) # พัก 3 วินาทีก่อนเรียกใหม่
                    if attempt == max_retries - 1:
                        print(f"      ❌ ข้ามหน้า {page_idx + 1} เนื่องจาก Error เกินกำหนด")
            
            full_markdown += f"\n{md_text}\n"
            
            # ลบไฟล์ Temp
            if os.path.exists(temp_path):
                os.remove(temp_path)
                    
        doc.close()
        
        print("✅ อ่านข้อมูลเสร็จสิ้น! กำลัง Extract...")
        
        parsed_data = parser.parse_markdown(full_markdown)
        # โยน file_type และกำหนด threshold 80% ให้ระบบปัดชื่อ
        flags_data = parser.validate_data(parsed_data, file_type, similarity_threshold=80)
        
        final_output = {
            "metadata": {"file_name": os.path.basename(test_pdf_path), "type": file_type},
            "extracted_data": parsed_data,
            "validation_flags": flags_data
        }
        
        display(Markdown("### 📊 ผลลัพธ์ที่ได้จากการเทส"))
        print(json.dumps(final_output, ensure_ascii=False, indent=4))
        
    except Exception as e:
        print(f"❌ เกิดข้อผิดพลาดในการเปิดไฟล์ PDF: {e}")

# Solution 2

Convert pdf file into W/B image (jpg), then CHUNKING the image into 2 parts to process.

This solution works with รายการเลือกตั้งแบบ**แบ่งเขต**

In [9]:
import os
import fitz
import tempfile
import time
from PIL import Image
from typhoon_ocr import ocr_document
from IPython.display import Markdown, display

# ใส่ชื่อไฟล์ที่อัปโหลดมา
test_pdf = "../อำเภอบ้านไร่/ตำบลแก่นมะกรูด/หน่วยเลือกตั้งที่ 1/ส.ส.5ทับ18.pdf" 

def process_image_chunk(img_chunk, chunk_name):
    """ฟังก์ชันผู้ช่วยสำหรับส่งรูปแต่ละท่อนไปให้ API"""
    with tempfile.NamedTemporaryFile(suffix=".jpg", delete=False) as tmp:
        # บีบอัดเป็นขาวดำ (L) ช่วยลดขนาด Payload ลง 3 เท่า
        img_chunk.convert('L').save(tmp.name, "JPEG", quality=70)
        
    print(f"   🚀 ส่ง {chunk_name} เข้า API...")
    start_time = time.time()
    
    try:
        result = ocr_document(pdf_or_image_path=tmp.name)
        print(f"   ✅ {chunk_name} เสร็จสิ้น (ใช้เวลา {time.time() - start_time:.2f} วินาที)")
    except Exception as e:
        print(f"   ❌ {chunk_name} Error: {e}")
        result = ""
        
    os.remove(tmp.name)
    return result

print(f"📄 เริ่มทดสอบไฟล์: {test_pdf}")
doc = fitz.open(test_pdf)

full_markdown = ""

# ลองเทสแค่หน้าแรกก่อน (หน้า 0)
page = doc.load_page(0)
pix = page.get_pixmap(dpi=150)

# แปลงเป็น PIL Image
img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
width, height = img.size

print(f"✂️ ทำการสับรูปภาพ (ขนาด {width}x{height}) ออกเป็น 2 ส่วน...")

# แบ่งครึ่งรูปภาพ (บน-ล่าง)
half_h = height // 2
top_chunk = img.crop((0, 0, width, half_h))
bottom_chunk = img.crop((0, half_h, width, height))

# ส่ง API ทีละส่วน
top_text = process_image_chunk(top_chunk, "ท่อนบน (ข้อมูลบัตร)")
time.sleep(2) # พักหายใจไม่ให้ API block
bottom_text = process_image_chunk(bottom_chunk, "ท่อนล่าง (ตารางคะแนน)")

# นำผลลัพธ์มาต่อกัน
full_markdown = top_text + "\n" + bottom_text

display(Markdown("### 🔍 ผลลัพธ์จากการต่อ Text (Raw Output)"))
print(full_markdown)

doc.close()

📄 เริ่มทดสอบไฟล์: ../อำเภอบ้านไร่/ตำบลแก่นมะกรูด/หน่วยเลือกตั้งที่ 1/ส.ส.5ทับ18.pdf
✂️ ทำการสับรูปภาพ (ขนาด 1240x1755) ออกเป็น 2 ส่วน...
   🚀 ส่ง ท่อนบน (ข้อมูลบัตร) เข้า API...
   ✅ ท่อนบน (ข้อมูลบัตร) เสร็จสิ้น (ใช้เวลา 2.80 วินาที)
   🚀 ส่ง ท่อนล่าง (ตารางคะแนน) เข้า API...
   ✅ ท่อนล่าง (ตารางคะแนน) เสร็จสิ้น (ใช้เวลา 10.67 วินาที)


### 🔍 ผลลัพธ์จากการต่อ Text (Raw Output)

ส.ส. ๕/๑๘

# รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขตเลือกตั้ง

สีขาว

ตามที่ได้มีพระราชกฤษฎีกาให้มีการเลือกตั้งสมาชิกสภาผู้แทนราษฎร และคณะกรรมการการเลือกตั้ง ได้กำหนดให้วันที่ 8 เดือน กุมภาพันธ์ พ.ศ. 2569 เป็นวันเลือกตั้ง นั้น

บัดนี้ คณะกรรมการประจำหน่วยเลือกตั้งได้ดำเนินการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขต เลือกตั้งของหน่วยเลือกตั้งที่ 1 หมู่ที่ 1 ตำบล/แขวง/เทศบาล แก่นมะกรูด อำเภอ/เขต มานไชย เขตเลือกตั้งที่ ๒๒ จังหวัดอุทัยธานี เสร็จสิ้นเป็นที่เรียบร้อยแล้ว ดังนั้น จึงขอรายงานผลการนับคะแนนของหน่วย เลือกตั้งดังกล่าว ดังนี้

๑. จำนวนผู้มีสิทธิเลือกตั้ง
๑.๑ จำนวนผู้มีสิทธิเลือกตั้งตามบัญชีรายชื่อผู้มีสิทธิเลือกตั้ง จำนวน 328 คน (สองร้อยแปดสิบแปด)
๑.๒ จำนวนผู้มีสิทธิเลือกตั้งที่มาแสดงตน (เฉพาะวันเลือกตั้ง) จำนวน 281 คน (สองร้อยแปดสิบเอ็ด)

๒. จำนวนบัตรเลือกตั้ง
๒.๑ จำนวนบัตรเลือกตั้งที่ได้รับจัดสรร จำนวน 340 บัตร (สามร้อยสี่สิบ)
๒.๒ จำนวนบัตรเลือกตั้งที่ใช้ จำนวน 281 บัตร (สองร้อยแปดสิบเอ็ด)
๒.๒.๑ บัตรดี จำนวน 254 บัตร (สองร้อยห้าสิบสี่)
๒.๒.๒ บัตรเสีย จำนวน 27 บัตร (สี่สิบ

In [10]:
display(Markdown(full_markdown))

ส.ส. ๕/๑๘

# รายงานผลการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขตเลือกตั้ง

สีขาว

ตามที่ได้มีพระราชกฤษฎีกาให้มีการเลือกตั้งสมาชิกสภาผู้แทนราษฎร และคณะกรรมการการเลือกตั้ง ได้กำหนดให้วันที่ 8 เดือน กุมภาพันธ์ พ.ศ. 2569 เป็นวันเลือกตั้ง นั้น

บัดนี้ คณะกรรมการประจำหน่วยเลือกตั้งได้ดำเนินการนับคะแนนสมาชิกสภาผู้แทนราษฎรแบบแบ่งเขต เลือกตั้งของหน่วยเลือกตั้งที่ 1 หมู่ที่ 1 ตำบล/แขวง/เทศบาล แก่นมะกรูด อำเภอ/เขต มานไชย เขตเลือกตั้งที่ ๒๒ จังหวัดอุทัยธานี เสร็จสิ้นเป็นที่เรียบร้อยแล้ว ดังนั้น จึงขอรายงานผลการนับคะแนนของหน่วย เลือกตั้งดังกล่าว ดังนี้

๑. จำนวนผู้มีสิทธิเลือกตั้ง
๑.๑ จำนวนผู้มีสิทธิเลือกตั้งตามบัญชีรายชื่อผู้มีสิทธิเลือกตั้ง จำนวน 328 คน (สองร้อยแปดสิบแปด)
๑.๒ จำนวนผู้มีสิทธิเลือกตั้งที่มาแสดงตน (เฉพาะวันเลือกตั้ง) จำนวน 281 คน (สองร้อยแปดสิบเอ็ด)

๒. จำนวนบัตรเลือกตั้ง
๒.๑ จำนวนบัตรเลือกตั้งที่ได้รับจัดสรร จำนวน 340 บัตร (สามร้อยสี่สิบ)
๒.๒ จำนวนบัตรเลือกตั้งที่ใช้ จำนวน 281 บัตร (สองร้อยแปดสิบเอ็ด)
๒.๒.๑ บัตรดี จำนวน 254 บัตร (สองร้อยห้าสิบสี่)
๒.๒.๒ บัตรเสีย จำนวน 27 บัตร (สี่สิบเจ็ด)
๒.๒.๓ บัตรที่ไม่เลือกผู้สมัครผู้ใด จำนวน 0 บัตร (ศูนย์)
๒.๓ จำนวนบัตรเลือกตั้งที่เหลือ จำนวน 59 บัตร (ห้าสิบเก้า)

๓. จำนวนคะแนนที่ผู้สมัครรับเลือกตั้งแต่ละคนได้รับเรียงตามลำดับหมายเลขประจำตัวผู้สมัคร

<table><tr><td>หมายเลขประจำตัว ผู้สมัคร</td><td>ชื่อตัว - ชื่อสกุล ผู้สมัครรับเลือกตั้ง</td><td>สังกัด พรรคการเมือง</td><td>ได้คะแนน (ให้กรอกทั้งตัวเลขและตัวอักษร)</td></tr><tr><td>๑</td><td>นายอรรถพล โต๋วสัจจา</td><td>เพื่อไทย</td><td>8 (แปด)</td></tr><tr><td>๒</td><td>นายชาดา ไทยเศรษฐ์</td><td>ภูมิใจไทย</td><td>193 (หนึ่งร้อยเก้าสิบสาม)</td></tr><tr><td>๓</td><td>นางสาวสุชาดา บัวพันธ์</td><td>ประชาชน</td><td>36 (สามสิบหก)</td></tr><tr><td>๔</td><td>ร.ต.ศึกษา ฟุ้งเฟื่อง ร.น.</td><td>ประชาธิปัตย์</td><td>17 (สิบเจ็ด)</td></tr><tr><td>๕</td><td>นางสุพรรณษา นันทา</td><td>กล้าธรรม</td><td>0 (ศูนย์)</td></tr><tr><td></td><td></td><td></td><td>(.................................................)</td></tr><tr><td></td><td></td><td></td><td>(.................................................)</td></tr><tr><td></td><td></td><td></td><td>(.................................................)</td></tr><tr><td></td><td></td><td></td><td>(.................................................)</td></tr><tr><td></td><td></td><td></td><td>(.................................................)</td></tr><tr><td colspan="2">รวมคะแนนทั้งสิ้น</td><td></td><td>254 (สองร้อยห้าสิบสี่)</td></tr></table>